# day-22-sdk-deep-dive — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 1 ----
def create_with_continue(client, max_retries=1, **kw):
    parts, kw = [], dict(kw)
    for _ in range(max_retries + 1):
        r = client.messages.create(**kw)
        parts.append("".join(b.text for b in r.content if b.type == "text"))
        if r.stop_reason != "max_tokens":
            break
        kw["max_tokens"] *= 2
        kw["messages"] = kw["messages"] + [{"role": "assistant", "content": parts[-1]},
                                           {"role": "user", "content": "continue"}]
    return "".join(parts), r.stop_reason

client.messages = MockMessages()
txt, stop = create_with_continue(client, model="claude-opus-5", max_tokens=1000,
                                 messages=[{"role": "user", "content": "hi"}])
print("S1:", stop, "| note: 'continue' can duplicate a partial word at the seam, or change")
print("    formatting mid-structure (broken JSON). Prefer a higher max_tokens up front, or")
print("    stream and stop when you have what you need.")

S1: end_turn | note: 'continue' can duplicate a partial word at the seam, or change
    formatting mid-structure (broken JSON). Prefer a higher max_tokens up front, or
    stream and stop when you have what you need.


In [9]:
# ---- Solution 4 ----
def render(msg):
    out = []
    for b in msg.content:
        if b.type == "text": out.append(b.text)
        elif b.type == "tool_use": out.append(f"[tool: {b.name}({json.dumps(b.input)})]")
        elif b.type == "thinking": out.append("[thinking ...]")
    return " ".join(out)

mixed = Message(id="m", model="claude-opus-5", stop_reason="tool_use",
                content=[TextBlock("Let me look that up."),
                         ToolUseBlock(id="tu_1", name="search", input={"q": "refunds"})])
print("S4:", render(mixed))

S4: Let me look that up. [tool: search({"q": "refunds"})]


### Solutions 2, 3, 5, 6 (sketch)

**S2:** `while est_tokens(self.messages) > budget and len(self.messages) > keep_recent: del
self.messages[0:2]`. Trim in pairs so the list stays `user`-first and role-alternating; never
touch `self.system`.

**S3:** a per-call timestamp anywhere in the cached prefix changes the bytes → the prefix no
longer matches → `cache_read_input_tokens` is 0 on every call and you pay full price forever.
Fix: move all volatile content (timestamps, request IDs, the question) *after* the last
`cache_control` breakpoint.

**S5:** our `_count` is words×4/3; `split()` is raw word count; the real tokenizer splits
sub-words, punctuation, and whitespace differently, counts tool-schema JSON and message
framing, and changed between model generations. Only `client.messages.count_tokens` matches
what you're billed.

**S6:** `r_a = client.messages.create(model=M, max_tokens=50, messages=msgs)` then
`r_b = client.messages.create(model=M, max_tokens=50, messages=msgs)` with the *same* `msgs`.
Nothing about `r_a` influences `r_b`; if you want continuity you must append `r_a`'s content to
`msgs` before the next call. That's the whole meaning of "stateless".

### Answer key
1. One — `POST /v1/messages`. Tools and `output_config` are fields on that request.
2. A plain string, or a list of content blocks (`text`, `image`, `document`, `tool_result`,
   …).
3. `content` is a list of typed blocks (text, thinking, tool_use); block 0 may not be text
   (e.g. a thinking or tool_use block first), so you filter by `.type`.
4. The model hit the output-token ceiling and was cut off mid-generation. You pay for the
   truncated partial and need another call to finish — so don't lowball `max_tokens`.
5. Resend the full message history every call (and append the assistant's own reply to it).
   Input tokens grow every turn, so long conversations need compaction or client-side
   trimming.
6. `temperature` / `top_p` / `top_k` (removed — use `effort`), and `thinking.budget_tokens`
   (removed — use `thinking: {type: "adaptive"}`). Also assistant prefill (a trailing
   assistant message) → use the system prompt or structured outputs.
7. A byte in the cached prefix changes between calls — a timestamp, UUID, unsorted JSON dump,
   or a varying tool list. Move all volatile content after the last `cache_control` breakpoint
   so the prefix is identical every time.